# ML preprocessing pipeline

Use `freshdata` to produce leakage-aware, typed, model-ready data, then train
a scikit-learn classifier.

```bash
pip install "freshdata[ml]" scikit-learn
```

In [ ]:
import numpy as np
import pandas as pd
import freshdata as fd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

## A raw, messy customer churn dataset

In [ ]:
rng = np.random.default_rng(3)
n = 800
tenure = rng.integers(1, 72, n)
monthly = rng.normal(70, 25, n)
churn = ((tenure < 12) & (monthly > 60)).astype(int)
raw = pd.DataFrame({
    'customer_id': range(n),
    'tenure_months': tenure.astype(float),
    'monthly_charges': monthly,
    'plan': rng.choice(['basic', 'pro', 'enterprise'], n),
    'churn': churn,
})
raw.loc[rng.choice(n, 70, replace=False), 'monthly_charges'] = np.nan
raw.head()

## Clean — protect the id and the target

Passing `target_column` ensures the label is never modified (no leakage), and
`id_columns` ensures identifiers are never imputed.

In [ ]:
clean_df, report = fd.clean(
    raw, id_columns=('customer_id',), target_column='churn', return_report=True
)
print(report.summary())
assert clean_df['churn'].equals(raw['churn'])  # target untouched

## Encode, split, and train on AI-ready data

In [ ]:
X = pd.get_dummies(clean_df.drop(columns=['customer_id', 'churn']))
y = clean_df['churn']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
model = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_tr, y_tr)
print(f'Test accuracy: {model.score(X_te, y_te):.3f}')

The same `fd.clean` call works inside any training pipeline — see the
[documentation](https://freshcode-org.github.io/freshdata/) for more.